In [1]:
# Notebook only: let Python find `agent.py` and the `utils/` and `tools/` folders.
# This notebook lives in 01_kyc_agent/modules/, so the 01_kyc_agent folder is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from models import model

# if __name__ == "__main__":
#     print(init_chat_model(model).invoke("Say hello to the KYC hackathon in one sentence.").content)

Hello, KYC hackathon—let’s build something great!


In [4]:
import importlib
import utils.state

importlib.reload(utils.state)

<module 'utils.state' from '/workspaces/finsights/agents/utils/state.py'>

In [3]:
from langchain.agents import create_agent
from utils.state import InvestigationContext

In [4]:
from utils.state import get_company, get_date

In [5]:
system_prompt=(project_root/ "supervisor_agent" / "prompt.md").read_text()
system_prompt

"You are a risk management assitant.\n\nEvery day you analyse the company you are invesitgating's price movements and then analyse the results using the tools provided.\n\nReview the data and produce a brief summary indicating what might be behind the price movements and if this "

In [41]:
from price_intelligence_agent.agent import agent as price_agent
from lseg_news_agent.agent import  agent as news_agent

In [16]:
@tool
def call_price_agent(
    query: str,
    runtime: ToolRuntime[InvestigationContext]
) -> str:
    """Delegate financial analysis tasks to the financial analyst."""

    result = price_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": query}
            ]
        },
        context=runtime.context
    )

    return result["messages"][-1].content

In [42]:
@tool
def call_news_agent(
    query: str,
    runtime: ToolRuntime[InvestigationContext]
) -> str:
    """Examine the news sources for potential market movement explanations."""

    result = news_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": query}
            ]
        },
        context=runtime.context
    )

    return result["messages"][-1].content

In [43]:
supervisor_agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    context_schema=InvestigationContext,
    tools=[get_company, get_date, call_price_agent, call_news_agent]
)

In [44]:
response = supervisor_agent.invoke(
{"messages": [{"role": "user", "content": "What can you tell me about price movements?"}]},
context=InvestigationContext(company_name='NVDA', date_str='2026-11-19')
)

print(response['messages'][-1].content[1]['text'])

KeyError: 'company_name'